In [1]:
"""
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
!wget {PREFIX}/01-agentic-rag/code/rag_helper.py
!wget {PREFIX}/04-evaluation/code/evaluation_utils.py
"""

import json
import cohere
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel
from embedder import Embedder
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index, VectorSearch

load_dotenv()
co = cohere.ClientV2()
embed = Embedder()

# Load documents and build chunks
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

# Load ground truth data
ground_truth = pd.read_csv('ground-truth.csv').to_dict('records')

#### Q1. Generating questions

In [2]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()


class Questions(BaseModel):
    questions: list[str]

def llm_structured_cohere(instructions, user_prompt, model="command-a-plus-05-2026"):
    """Cohere version of structured output - returns parsed result and token usage"""
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]
    
    # Add instruction to return JSON
    json_instruction = "\n\nReturn your response as a JSON object with a 'questions' field containing a list of question strings."
    messages[-1]["content"] += json_instruction
    
    response = co.chat(
        model=model,
        messages=messages,
        response_format={"type": "json_object"},
    )
    
    # Extract text content
    response_text = next(item.text for item in response.message.content if hasattr(item, "text"))
    result = Questions(**json.loads(response_text))
    return result, response.usage.tokens

# Generate questions for the first 3 pages
target_pages = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md", 
    "01-agentic-rag/lessons/03-rag.md"
]

# Create a mapping from filename to document
doc_map = {doc['filename']: doc for doc in documents}

usages = []
ground_truth_gen = []

for page in target_pages:
    doc = doc_map[page]
    user_prompt = f"Filename: {doc['filename']}\n\nContent:\n{doc['content']}"
    result, usage = llm_structured_cohere(data_gen_instructions, user_prompt)
    usages.append(usage)
    for q in result.questions:
        ground_truth_gen.append({"question": q, "filename": page})

avg_input_tokens = sum(u.input_tokens for u in usages) / len(usages)
print(f"Average input tokens: {avg_input_tokens}")

Average input tokens: 1222.6666666666667


#### Q2. First result with text search

In [3]:
chunks = chunk_documents(documents, size=2000, step=1000)

# Create text index
text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
text_index.fit(chunks)

def text_search(query, num_results = 5):
    """Search documents by keyword text matching."""
    return text_index.search(query, num_results=num_results)

q = ground_truth_gen[0]["question"]
print(f"Question: {q}")

# Run text search for the first question
text_results = text_search(q, num_results=5)
print(f"First result filename: {text_results[0]['filename']}")

Question: What is the main goal of building a RAG system in this course?
First result filename: 01-agentic-rag/lessons/04-dataset.md


#### Q3. First result with vector search

In [4]:
# Create vector index
chunk_texts = [chunk['content'] for chunk in chunks]
X = embed.encode_batch(chunk_texts)

vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(X, chunks)

def vector_search(query, num_results = 5):
    """Search documents by dense vector cosine similarity."""
    query_vector = embed.encode(query)
    return vector_index.search(query_vector, num_results=num_results)

# Run vector search for the same question
vector_results = vector_search(q, num_results=5)
print(f"First vector result filename: {vector_results[0]['filename']}")

First vector result filename: 04-evaluation/lessons/14-agent-evaluation.md


#### Q4. Evaluating text search

In [5]:
def compute_relevance(question, ground_truth_filename, search_function, num_results=5):
    """Run search for a question and return a list of 0s and 1s indicating relevance."""
    results = search_function(question, num_results=num_results)
    return [1 if result["filename"] == ground_truth_filename else 0 for result in results]


def hit_rate(relevance_list):
    """Fraction of questions where the correct page appears in the results."""
    hits = sum(any(relevance) for relevance in relevance_list)
    return hits / len(relevance_list)


def mrr(relevance_list):
    """Mean Reciprocal Rank - rewards finding the page near the top."""
    reciprocal_ranks = []
    for relevance in relevance_list:
        for rank, is_relevant in enumerate(relevance, start=1):
            if is_relevant:
                reciprocal_ranks.append(1 / rank)
                break
        else:
            reciprocal_ranks.append(0)
    return sum(reciprocal_ranks) / len(reciprocal_ranks)


def evaluate(ground_truth, search_function, num_results=5):
    """Run a search function over the whole ground truth and return both metrics."""
    relevance_list = [
        compute_relevance(record["question"], record["filename"], search_function, num_results=num_results)
        for record in ground_truth
    ]
    return {
        "hit_rate": hit_rate(relevance_list),
        "mrr": mrr(relevance_list),
    }

text_search_metrics = evaluate(ground_truth, text_search, num_results=5)
print(f"Hit Rate: {text_search_metrics['hit_rate']}")

Hit Rate: 0.7583333333333333


#### Q5. Evaluating vector search

In [6]:
print(f"MRR: {text_search_metrics['mrr']}")

MRR: 0.5942592592592593


#### Q6. Tuning hybrid search


In [7]:
def rrf(result_lists, k=60, num_results=5):
    """Reciprocal Rank Fusion to merge and rank multiple retrieval result lists."""
    scores, docs = {}, {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60, num_results=5):
    """Combine text and vector search results using Reciprocal Rank Fusion."""
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k, num_results=num_results)
    
k_values = [1, 5, 10, 25, 50, 100]
results = {}

for k in k_values:
    search_func = lambda q, num_results=5, k=k: hybrid_search(q, k=k, num_results=num_results)
    metrics = evaluate(ground_truth, search_func, num_results=5)
    results[k] = metrics["mrr"]
    print(f"k={k}: MRR = {metrics['mrr']}")

best_k = max(results, key=results.get)
print(f"\nBest k value: {best_k} with MRR = {results[best_k]}")

k=1: MRR = 0.6481944444444444
k=5: MRR = 0.6419444444444444
k=10: MRR = 0.6402314814814815
k=25: MRR = 0.6379166666666667
k=50: MRR = 0.6379166666666667
k=100: MRR = 0.6379166666666667

Best k value: 1 with MRR = 0.6481944444444444
